# Exercises XP: RAG with LangChain

Système de questions-réponses basé sur la **Retrieval-Augmented Generation (RAG)** avec LangChain, FAISS et un modèle Hugging Face local.

## 0) Setup

In [ ]:
# Installer toutes les dépendances nécessaires
!pip -q install -U datasets transformers sentence-transformers faiss-cpu \
    langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline

# Note : RetrievalQA a été déplacé dans langchain.chains sur les versions récentes
try:
    from langchain.chains import RetrievalQA
except ImportError:
    from langchain_community.chains import RetrievalQA

print("✅ Imports OK")

## 1) Load dataset and convert to Documents

In [ ]:
# Explorer le dataset avant de charger
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

print("Colonnes disponibles :", ds.column_names)
print("\nExemple de ligne brute :")
print(ds[0])

In [ ]:
# Convertir les lignes du dataset en objets Document LangChain
# - page_content : le texte principal du document
# - metadata : dictionnaire avec la source (utile pour les citations et le débogage)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],           # Texte principal
            metadata={"source": row[source_column], "doc_id": i}  # Métadonnées
        )
    )

print("Documents chargés :", len(documents))
print("\nMétadonnées du 1er document :", documents[0].metadata)
print("\nDébut du contenu :")
print(documents[0].page_content[:350])

## 2) Split into chunks

**Pourquoi découper ?**  
Les modèles d'embedding ont une limite de tokens, et les chunks plus petits permettent une récupération plus précise.

**Estimation des paramètres :**  
- Longueur moyenne d'un document ≈ 2000 caractères → `chunk_size = 512` (≈ 4 chunks par doc)  
- Chevauchement ≈ 10–20% du chunk_size → `chunk_overlap = 64`

In [ ]:
# Estimer la longueur moyenne des documents
avg_len = sum(len(d.page_content) for d in documents) / len(documents)
print(f"Longueur moyenne des documents : {avg_len:.0f} caractères")

In [ ]:
# Configuration 1 (valeur par défaut recommandée)
chunk_size = 512       # ~4 chunks par document en moyenne
chunk_overlap = 64     # ~12% de chevauchement pour préserver le contexte entre chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

splits = splitter.split_documents(documents)
print(f"✅ Configuration 1 — chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
print(f"   Nombre de chunks : {len(splits)}")
print(f"\nPremier chunk — métadonnées : {splits[0].metadata}")
print(splits[0].page_content[:350])

In [ ]:
# Expérimentation : tester d'autres paramètres pour observer l'impact

# Configuration 2 — chunks plus grands, moins nombreux
splitter_large = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)
splits_large = splitter_large.split_documents(documents)
print(f"Config 2 (chunk_size=1024) → {len(splits_large)} chunks")

# Configuration 3 — chunks plus petits, plus nombreux
splitter_small = RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=32)
splits_small = splitter_small.split_documents(documents)
print(f"Config 3 (chunk_size=256)  → {len(splits_small)} chunks")

print("\n💡 Remarque : des chunks plus petits = récupération plus précise mais contexte réduit.")
print("   Des chunks plus grands = plus de contexte mais moins de précision dans la récupération.")

## 3) Vector store + retriever (FAISS)

In [ ]:
# Modèle d'embedding : all-MiniLM-L6-v2 (384 dimensions, rapide et efficace)
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

# Créer le vectorstore FAISS avec similarité cosinus
vectorstore = FAISS.from_documents(
    splits,                                      # Les chunks à indexer
    embeddings,                                  # Le modèle d'embedding
    distance_strategy=DistanceStrategy.COSINE    # Métrique de similarité
)

# Retriever par défaut : k=4 documents les plus proches
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✅ Retriever prêt (k=4)")

In [ ]:
# Expérimentation : tester différentes valeurs de k
test_question = "How can I retrieve a model from the Hugging Face Hub?"

for k in [2, 4, 6]:
    r = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = r.get_relevant_documents(test_question)
    print(f"\nk={k} → {len(docs)} documents récupérés")
    for d in docs:
        print(f"  Source: {d.metadata.get('source', 'N/A')}")

print("\n💡 k plus grand = meilleure couverture mais plus de bruit potentiel.")

## 4 bis) Vérification de cohérence de la récupération ✅

Avant de brancher le LLM, on vérifie que les chunks récupérés sont bien pertinents.

In [ ]:
# Vérification de la pertinence des chunks récupérés
sanity_question = "How can I retrieve a model from the Hugging Face Hub?"
retrieved_docs = retriever.get_relevant_documents(sanity_question)

print(f"Question : '{sanity_question}'")
print(f"\n{len(retrieved_docs)} chunks récupérés :\n")

for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source : {doc.metadata.get('source', 'N/A')}")
    print(doc.page_content[:300])
    print()

print("✅ Si les chunks ci-dessus mentionnent le Hub, la récupération est cohérente.")
print("   Sinon, ajustez chunk_size, chunk_overlap ou k avant de continuer.")

## 4) Build the RAG chain

In [ ]:
from transformers import pipeline as hf_pipeline
from langchain_huggingface import HuggingFacePipeline

try:
    from langchain.chains import RetrievalQA
except ImportError:
    from langchain_community.chains import RetrievalQA

# Modèle LLM local : flan-t5-small (léger, pas de clé API requise)
llm_id = "google/flan-t5-small"

hf = hf_pipeline(
    task="text2text-generation",   # flan-t5 est un modèle seq2seq
    model=llm_id,
    max_new_tokens=256,
    do_sample=False                # Génération déterministe (greedy)
)

# Wrapper LangChain autour du pipeline HuggingFace
llm = HuggingFacePipeline(pipeline=hf)

# Construction de la chaîne RAG
# chain_type="stuff" : concatène tous les chunks récupérés dans le prompt
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True   # Pour afficher les sources des chunks utilisés
)

print("✅ Chaîne RAG prête")

## 5) Demo: RAG vs no-RAG

Comparaison entre la réponse du LLM seul (sans contexte) et la réponse augmentée par la récupération.

In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# ── No-RAG : LLM seul, sans contexte externe ─────────────────────────────────
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# ── RAG : LLM + contexte récupéré depuis FAISS ───────────────────────────────
rag_result = qa({"query": q})  # On passe la question via la clé "query"

# ── Affichage comparatif ──────────────────────────────────────────────────────
print("=" * 60)
print(f"Q: {q}")
print("=" * 60)
print("\n🔴 No-RAG answer (LLM seul) :")
print(" ", no_rag_answer)

print("\n🟢 RAG answer (LLM + contexte récupéré) :")
print(" ", rag_result["result"])

print("\n📚 Sources utilisées par le RAG :")
for d in rag_result["source_documents"]:
    print("  -", d.metadata.get("source", "N/A"))

In [ ]:
# Tester plusieurs questions supplémentaires
questions = [
    "What is the Hugging Face Hub?",
    "How do I fine-tune a transformer model?",
    "What are datasets in Hugging Face?"
]

for question in questions:
    print("\n" + "=" * 60)
    print(f"Q: {question}")
    result = qa({"query": question})
    print(f"A: {result['result']}")
    print("Sources :")
    for d in result["source_documents"]:
        print("  -", d.metadata.get("source", "N/A"))

---
## ✅ Récapitulatif — Ce que vous avez appris

| Étape | Concept clé |
|-------|-------------|
| Dataset → Documents | Structurer des données brutes pour LangChain |
| Chunking | `chunk_size` et `chunk_overlap` influencent la précision de la récupération |
| FAISS + embeddings | Stocker et rechercher des vecteurs sémantiques |
| Vérification de cohérence | Toujours inspecter les chunks avant de brancher le LLM |
| `RetrievalQA` | Pipeline RAG complet : retrieve → stuff → generate |
| RAG vs no-RAG | Le RAG ancre les réponses dans des documents réels |

**💡 Pistes d'amélioration :**
- Tester un modèle plus puissant (`flan-t5-base` ou `flan-t5-large`)
- Utiliser `chain_type="map_reduce"` pour de longs contextes
- Ajouter un `PromptTemplate` personnalisé pour mieux guider le LLM